# 🅿️ ParkVision – Live Demo Notebook
**ITAI 1378 – Computer Vision & AI**  
This notebook loads your trained `best.pt` model and runs it on parking lot images with a red/green visual overlay showing occupied vs. empty spaces.

---

## Section 1 – Setup & Load Model

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python-headless matplotlib Pillow -q

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Mount Google Drive to access your saved model
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from ultralytics import YOLO
import os

# -------------------------------------------------------
# OPTION A: Load from Google Drive (recommended)
# Update this path to wherever you saved best.pt
# -------------------------------------------------------
MODEL_PATH = '/content/drive/MyDrive/ParkVision/runs/detect/ParkVision/train_v1/weights/best.pt'

# -------------------------------------------------------
# OPTION B: Upload best.pt directly to Colab
# Uncomment these lines if you're uploading manually
# -------------------------------------------------------
# from google.colab import files
# uploaded = files.upload()  # Upload best.pt
# MODEL_PATH = 'best.pt'

# Load the model
model = YOLO(MODEL_PATH)
print('Model loaded successfully!')
print('Classes:', model.names)

## Section 2 – Run on Test Images with Visual Overlay

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import random
import time

# Color scheme
COLOR_EMPTY    = (0, 220, 0)    # Green for empty spots
COLOR_OCCUPIED = (0, 0, 220)    # Red for occupied spots  
COLOR_TEXT     = (255, 255, 255) # White text
FONT = cv2.FONT_HERSHEY_DUPLEX

def run_parkvision(image_path, confidence=0.25):
    """
    Run ParkVision model on a single image.
    Returns annotated image + counts dict.
    """
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Run inference and time it
    start = time.time()
    results = model(img_rgb, conf=confidence, verbose=False)
    elapsed_ms = (time.time() - start) * 1000
    
    counts = {'empty': 0, 'occupied': 0}
    annotated = img_rgb.copy()
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cls_id = int(box.cls[0].item())
            conf_score = float(box.conf[0].item())
            label = model.names[cls_id]
            
            # Choose color based on class
            color = COLOR_EMPTY if label == 'empty' else COLOR_OCCUPIED
            counts[label] = counts.get(label, 0) + 1
            
            # Draw filled semi-transparent rectangle
            overlay = annotated.copy()
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
            cv2.addWeighted(overlay, 0.25, annotated, 0.75, 0, annotated)
            
            # Draw border
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            
            # Draw confidence label
            label_text = f'{label[0].upper()} {conf_score:.0%}'
            cv2.putText(annotated, label_text, (x1 + 4, y1 + 16),
                        FONT, 0.4, COLOR_TEXT, 1, cv2.LINE_AA)
    
    # Draw summary HUD in top-left corner
    total = counts.get('empty', 0) + counts.get('occupied', 0)
    avail = counts.get('empty', 0)
    pct   = (avail / total * 100) if total > 0 else 0
    
    hud_lines = [
        f"ParkVision",
        f"Empty:    {counts.get('empty', 0)}",
        f"Occupied: {counts.get('occupied', 0)}",
        f"Available: {pct:.0f}%",
        f"Inference: {elapsed_ms:.1f}ms"
    ]
    
    # HUD background
    hud_h = len(hud_lines) * 26 + 16
    cv2.rectangle(annotated, (8, 8), (230, hud_h), (20, 20, 20), -1)
    cv2.rectangle(annotated, (8, 8), (230, hud_h), (80, 80, 80), 1)
    
    for i, line in enumerate(hud_lines):
        color = (150, 255, 150) if 'Empty' in line else \
                (255, 120, 120) if 'Occupied' in line else \
                (255, 220, 80) if 'Available' in line else \
                (200, 200, 255) if 'ParkVision' in line else (220, 220, 220)
        cv2.putText(annotated, line, (16, 32 + i * 26),
                    FONT, 0.55, color, 1, cv2.LINE_AA)
    
    return annotated, counts, elapsed_ms

print('Demo functions ready!')

In [ ]:
# -------------------------------------------------------
# Run on test images from your dataset
# -------------------------------------------------------
DATASET_PATH = '/content/drive/MyDrive/ParkVision/dataset'

# Find test images
test_images = glob.glob(os.path.join(DATASET_PATH, 'test/images/*.jpg'))
if not test_images:
    # Fallback to valid set
    test_images = glob.glob(os.path.join(DATASET_PATH, 'valid/images/*.jpg'))

print(f'Found {len(test_images)} test images')

# Pick 6 random samples
samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('ParkVision – Live Detection Results', 
             fontsize=20, color='white', fontweight='bold', y=1.01)

total_empty = 0
total_occ = 0

for ax, img_path in zip(axes.flatten(), samples):
    annotated, counts, ms = run_parkvision(img_path)
    total_empty += counts.get('empty', 0)
    total_occ   += counts.get('occupied', 0)
    
    ax.imshow(annotated)
    ax.set_title(f"E:{counts.get('empty',0)}  O:{counts.get('occupied',0)}  |  {ms:.0f}ms",
                 color='white', fontsize=11)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

plt.tight_layout()
plt.savefig('/content/demo_grid.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()

print(f'\nTotal across {len(samples)} images:')
print(f'  Empty spaces:    {total_empty}')
print(f'  Occupied spaces: {total_occ}')
print(f'  Availability:    {total_empty/(total_empty+total_occ)*100:.1f}%')

## Section 3 – Run on a Single Image (Great for Live Demo)

In [ ]:
# Pick any single image and display it large — perfect for your live demo
demo_image = random.choice(test_images)
print(f'Running on: {os.path.basename(demo_image)}')

annotated, counts, ms = run_parkvision(demo_image, confidence=0.25)

plt.figure(figsize=(16, 10))
plt.imshow(annotated)
plt.title(
    f"ParkVision Detection  |  Empty: {counts.get('empty',0)}  "
    f"Occupied: {counts.get('occupied',0)}  |  {ms:.1f}ms inference",
    fontsize=14, color='white', pad=12
)
plt.axis('off')
plt.gca().set_facecolor('#1a1a2e')
plt.gcf().set_facecolor('#1a1a2e')
plt.tight_layout()
plt.savefig('/content/demo_single.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()

## Section 4 – Upload Your Own Image

In [ ]:
# Upload ANY parking lot image from your computer and run ParkVision on it
from google.colab import files
from PIL import Image
import io

print('Select a parking lot image to upload...')
uploaded = files.upload()

for filename, data in uploaded.items():
    # Save to disk
    with open(f'/content/{filename}', 'wb') as f:
        f.write(data)
    
    print(f'Running ParkVision on {filename}...')
    annotated, counts, ms = run_parkvision(f'/content/{filename}')
    
    plt.figure(figsize=(16, 10))
    plt.imshow(annotated)
    plt.title(
        f"ParkVision  |  Empty: {counts.get('empty',0)}  "
        f"Occupied: {counts.get('occupied',0)}  |  {ms:.1f}ms",
        fontsize=14, color='white'
    )
    plt.axis('off')
    plt.gcf().set_facecolor('#1a1a2e')
    plt.tight_layout()
    plt.savefig(f'/content/custom_{filename}.png', dpi=150, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.show()
    print(f'Saved: /content/custom_{filename}.png')

## Section 5 – Speed Benchmark

In [ ]:
# Measure average inference speed across 20 images
benchmark_images = random.sample(test_images, min(20, len(test_images)))
times = []

print('Running speed benchmark...')
for img_path in benchmark_images:
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    start = time.time()
    model(img_rgb, conf=0.25, verbose=False)
    times.append((time.time() - start) * 1000)

avg_ms  = sum(times) / len(times)
max_ms  = max(times)
min_ms  = min(times)
fps     = 1000 / avg_ms

print(f'\nSpeed Benchmark Results ({len(benchmark_images)} images):')
print(f'  Average inference: {avg_ms:.1f}ms')
print(f'  Fastest:           {min_ms:.1f}ms')
print(f'  Slowest:           {max_ms:.1f}ms')
print(f'  Throughput:        {fps:.1f} FPS')
print(f'  Target (<1000ms):  {"PASSED" if avg_ms < 1000 else "FAILED"}')
print(f'  Speedup vs target: {1000/avg_ms:.0f}x faster')

# Plot distribution
plt.figure(figsize=(10, 4))
plt.hist(times, bins=10, color='#6c63ff', edgecolor='white', alpha=0.85)
plt.axvline(avg_ms, color='#ff6584', linewidth=2, label=f'Mean: {avg_ms:.1f}ms')
plt.axvline(1000,   color='orange',   linewidth=2, linestyle='--', label='Target: 1000ms')
plt.title('Inference Speed Distribution', fontsize=14, color='white')
plt.xlabel('Time (ms)', color='white')
plt.ylabel('Count', color='white')
plt.legend(facecolor='#1a1a2e', labelcolor='white')
plt.gca().set_facecolor('#0f0f23')
plt.gcf().set_facecolor('#1a1a2e')
plt.tick_params(colors='white')
plt.tight_layout()
plt.savefig('/content/speed_benchmark.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()

## Section 6 – Save Demo Outputs to Google Drive

In [ ]:
import shutil

DEMO_OUTPUT_DIR = '/content/drive/MyDrive/ParkVision/demo_outputs'
os.makedirs(DEMO_OUTPUT_DIR, exist_ok=True)

files_to_save = [
    '/content/demo_grid.png',
    '/content/demo_single.png',
    '/content/speed_benchmark.png',
]

for f in files_to_save:
    if os.path.exists(f):
        dest = os.path.join(DEMO_OUTPUT_DIR, os.path.basename(f))
        shutil.copy(f, dest)
        print(f'Saved: {dest}')
    else:
        print(f'Not found (run that section first): {f}')

print('\nAll demo outputs saved to Google Drive!')
print('Use these screenshots in your presentation slides.')

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `best.pt` not found | Check the `MODEL_PATH` in Section 1 matches where you saved it in Drive |
| 0 detections | Lower confidence: `run_parkvision(img, confidence=0.15)` |
| Wrong class names | Check `model.names` output — class IDs might be swapped |
| Drive not mounting | Re-run the `drive.mount` cell and re-authorize |
| Colab disconnects | Save outputs to Drive frequently — Section 6 covers this |
